In [1]:
#Import Libraries

import pandas as pd 
import numpy as np
import os 
import glob
import requests
import json
from datetime import datetime

In [2]:
path=r"C:\Users\neuma\OneDrive\Desktop\Specialisation\NY_CityBike_Analysis\2022-citibike-tripdata\2022 Tripdata"

In [3]:
#create a list of files, instead of reading in single files

csv_files = glob.glob(os.path.join(path, "*.csv"))

In [4]:
#using concat

df=[pd.read_csv(file, low_memory=False) for file in csv_files]
citibike_2022=pd.concat(df, ignore_index=True)

In [5]:
citibike_2022.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,BFD29218AB271154,electric_bike,2022-01-21 13:13:43.392,2022-01-21 13:22:31.463,West End Ave & W 107 St,7650.05,Mt Morris Park W & W 120 St,7685.14,40.802117,-73.968181,40.804038,-73.945925,member
1,7C953F2FD7BE1302,classic_bike,2022-01-10 11:30:54.162,2022-01-10 11:41:43.422,4 Ave & 3 St,4028.04,Boerum Pl\t& Pacific St,4488.09,40.673746,-73.985649,40.688489,-73.991160,member
2,95893ABD40CED4B8,electric_bike,2022-01-26 10:52:43.096,2022-01-26 11:06:35.227,1 Ave & E 62 St,6753.08,5 Ave & E 29 St,6248.06,40.761227,-73.960940,40.745168,-73.986831,member
3,F853B50772137378,classic_bike,2022-01-03 08:35:48.247,2022-01-03 09:10:50.475,2 Ave & E 96 St,7338.02,5 Ave & E 29 St,6248.06,40.783964,-73.947167,40.745168,-73.986831,member
4,7590ADF834797B4B,classic_bike,2022-01-22 14:14:23.043,2022-01-22 14:34:57.474,6 Ave & W 34 St,6364.10,5 Ave & E 29 St,6248.06,40.749640,-73.988050,40.745168,-73.986831,member


In [6]:
citibike_2022.shape

(29838806, 13)

## Comment 
### Code: pd.concat + glob

Glob is the easiest and most efficient way to automatically collect all csv files at once, instead of having to type/ read in every file by itself. 
Pd.concat stitches the datatsets along the row axis and resetting the index.

## Weather Data using NOAA's API

In [40]:
Token='ENRKWnSoZtgFSBqYadjCqzWMFOQSRGdV'

In [55]:
r=requests.get(
    "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
    "?datasetid=GHCND"
    "&stationid=GHCND:USW00094846"
    "&startdate=2022-01-01"
    "&enddate=2022-12-31"
    "&limit=1000",
    headers={'token':Token}
)

In [56]:
d=json.loads(r.text)

In [57]:
d

{'metadata': {'resultset': {'offset': 1, 'count': 6802, 'limit': 1000}},
 'results': [{'date': '2022-01-01T00:00:00',
   'datatype': 'ADPT',
   'station': 'GHCND:USW00094846',
   'attributes': ',,W,',
   'value': -28},
  {'date': '2022-01-01T00:00:00',
   'datatype': 'ASLP',
   'station': 'GHCND:USW00094846',
   'attributes': ',,W,',
   'value': 10105},
  {'date': '2022-01-01T00:00:00',
   'datatype': 'ASTP',
   'station': 'GHCND:USW00094846',
   'attributes': ',,W,',
   'value': 9858},
  {'date': '2022-01-01T00:00:00',
   'datatype': 'AWBT',
   'station': 'GHCND:USW00094846',
   'attributes': ',,W,',
   'value': -11},
  {'date': '2022-01-01T00:00:00',
   'datatype': 'AWND',
   'station': 'GHCND:USW00094846',
   'attributes': ',,W,',
   'value': 88},
  {'date': '2022-01-01T00:00:00',
   'datatype': 'PRCP',
   'station': 'GHCND:USW00094846',
   'attributes': ',,W,2400',
   'value': 58},
  {'date': '2022-01-01T00:00:00',
   'datatype': 'RHAV',
   'station': 'GHCND:USW00094846',
   'attri

In [58]:
results=d["results"]
df_temp=pd.DataFrame(results)

df_temp.head()

,date,datatype,station,attributes,value
0,2022-01-01T00:00:00,ADPT,GHCND:USW00094846,",,W,",-28
1,2022-01-01T00:00:00,ASLP,GHCND:USW00094846,",,W,",10105
2,2022-01-01T00:00:00,ASTP,GHCND:USW00094846,",,W,",9858
3,2022-01-01T00:00:00,AWBT,GHCND:USW00094846,",,W,",-11
4,2022-01-01T00:00:00,AWND,GHCND:USW00094846,",,W,",88


In [59]:
df_temp.shape

(1000, 5)

### Merge

In [60]:
df_temp = df_temp[df_temp["datatype"] == "TAVG"].copy()
df_temp["date"] = pd.to_datetime(df_temp["date"]).dt.normalize()
df_temp["avgTemp"] = df_temp["value"] / 10
df_temp = df_temp[["date", "avgTemp"]]

In [61]:
citibike_2022["date"] = pd.to_datetime(citibike_2022["started_at"]).dt.normalize()
df_merged = citibike_2022.merge(df_temp, on="date", how="left")

In [62]:
df_merged.shape

(29838806, 15)

In [63]:
#saving

df_temp.to_csv("df_temp.csv", index=False)

In [69]:
df_merged.to_csv("citibike_merged.csv", index=False)

In [70]:
df_merged.to_csv(r"C:\Users\neuma\OneDrive\Desktop\Specialisation\NY_CityBike_Analysis\venv\df_merged", index=False)

In [72]:
df_temp.to_csv(r"C:\Users\neuma\OneDrive\Desktop\Specialisation\NY_CityBike_Analysis\venv\df_temp")